# 01. EDA: данные для дообучения

Ноутбук отвечает на вопросы из требований к курсовой: **откуда данные, как получены, сколько объектов и признаков, какие есть проблемы**, и проверяет, что разбиение корректно и нет утечек.

План:
1. Происхождение и состав GSM8K
2. Структура решений и ответов
3. Длины в токенах (и сколько примеров не влезет в `max_seq_len`)
4. Проблемы данных: дубликаты, пересечения, формат
5. Разбиение train/val, вложенные подвыборки, утечки
6. (Опционально) MetaMathQA
7. Выводы

In [ ]:
import os, sys, json, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

# корень репозитория: ноутбук можно запускать и из notebooks/, и из корня
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

FIG = ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

from datasets import load_dataset
from transformers import AutoTokenizer
from src.config import load_config
from src.data import build_datasets

cfg = load_config(["configs/base.yaml"])
tok = AutoTokenizer.from_pretrained(cfg["model"]["name"])
TEMPLATE = cfg["data"]["prompt_template"]
MAX_LEN = cfg["data"]["max_seq_len"]
print("Модель/токенизатор:", cfg["model"]["name"], "| max_seq_len:", MAX_LEN)

## 1. Происхождение и состав

GSM8K: ~8.5 тыс. текстовых школьных математических задач с решением по шагам. Итоговый ответ в решении стоит после `####`.
Поле лицензии в метаданных датасета часто пустое, поэтому **лицензию нужно сверить на странице датасета** на Hugging Face и записать в `docs/data_card.md`.

In [ ]:
raw_train = load_dataset("openai/gsm8k", "main", split="train")
raw_test = load_dataset("openai/gsm8k", "main", split="test")

print("train:", len(raw_train), "| test:", len(raw_test))
print("признаки:", raw_train.column_names)
print("версия:", raw_train.info.version, "| поле license в метаданных:", repr(raw_train.info.license))
print("\n--- пример ---")
print("Q:", raw_train[0]["question"])
print("A:", raw_train[0]["answer"])

## 2. Структура решений и ответов

Для каждой задачи считаем: число шагов решения (строк до `####`), наличие аннотаций калькулятора `<<...>>`, итоговый ответ и его тип. Тип ответа важен для метрики: сравнение идёт по извлечённому **числу**, поэтому нужно знать долю дробных и отрицательных ответов.

In [ ]:
def tok_len(texts):
    return [len(x) for x in tok(list(texts), add_special_tokens=False)["input_ids"]]

def answer_type(s):
    try:
        v = float(s)
    except ValueError:
        return "не число"
    if v < 0:
        return "отрицательное"
    return "дробное" if v != int(v) else "целое"

def parse_split(ds, name):
    df = ds.to_pandas().rename(columns={"answer": "solution"})
    df["split"] = name
    df["has_marker"] = df.solution.str.contains("####", regex=False)
    df["final"] = df.solution.str.split("####").str[-1].str.strip().str.replace(",", "", regex=False)
    df["answer_type"] = df.final.map(answer_type)
    df["steps"] = df.solution.map(lambda s: sum(1 for l in s.split("\n") if l.strip() and not l.startswith("####")))
    df["calc_annot"] = df.solution.str.contains("<<", regex=False)
    df["q_tok"] = tok_len(df.question)
    df["s_tok"] = tok_len(" " + df.solution)
    # длина обучающего примера как в train.py: промпт + " " + решение + EOS
    df["total_tok"] = tok_len([TEMPLATE.format(question=q) for q in df.question]) + df.s_tok + 1
    return df

df = pd.concat([parse_split(raw_train, "train"), parse_split(raw_test, "test")], ignore_index=True)
df.groupby("split")[["q_tok", "s_tok", "total_tok", "steps"]].describe(percentiles=[0.5, 0.95]).T

In [ ]:
summary = {}
summary["n"] = df.groupby("split").size().to_dict()
summary["no_marker"] = int((~df.has_marker).sum())
summary["answer_type_share"] = df.groupby("split").answer_type.value_counts(normalize=True).round(4).unstack().to_dict()
summary["calc_annot_share"] = df.groupby("split").calc_annot.mean().round(4).to_dict()
summary["share_over_max_len"] = df.assign(over=df.total_tok > MAX_LEN).groupby("split").over.mean().round(4).to_dict()
print(json.dumps(summary, indent=1, ensure_ascii=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
colors = {"train": "#1f77b4", "test": "#ff7f0e"}

ax = axes[0, 0]
for s, d in df.groupby("split"):
    ax.hist(d.total_tok, bins=50, alpha=0.55, density=True, label=s, color=colors[s])
ax.axvline(MAX_LEN, color="k", ls="--", lw=1, label=f"max_seq_len={MAX_LEN}")
ax.set(title="Длина примера (промпт + решение), токены", xlabel="токены", ylabel="плотность")
ax.legend()

ax = axes[0, 1]
steps = df.groupby(["split", "steps"]).size().unstack(0).fillna(0)
(steps / steps.sum()).plot.bar(ax=ax, color=[colors["test"], colors["train"]])
ax.set(title="Число шагов решения", xlabel="шагов", ylabel="доля")

ax = axes[1, 0]
vals = pd.to_numeric(df.final, errors="coerce")
mag = np.log10(vals[(vals != 0) & vals.notna()].abs())
ax.hist(mag, bins=40, color="#2ca02c", alpha=0.8)
ax.set(title="Величина ответа, log10|ответ|", xlabel="log10|ответ|", ylabel="число задач")

ax = axes[1, 1]
(df.groupby("split").answer_type.value_counts(normalize=True).unstack(0) * 100).plot.bar(ax=ax, color=[colors["test"], colors["train"]])
ax.set(title="Тип итогового ответа", xlabel="", ylabel="%")
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(FIG / "eda_gsm8k_overview.png", bbox_inches="tight")
plt.show()

**Что смотреть на графиках.**
- Хвост длин за `max_seq_len`: такие примеры отбрасываются в `tokenize_rows`. Если их заметная доля, увеличьте `max_seq_len` или честно напишите о смещении.
- Распределение шагов и величин ответов у train и test должно быть похоже; иначе на тесте модель сталкивается с другим распределением.
- Дробных и отрицательных ответов мало, поэтому сравнение «число против числа» в метрике почти не страдает от формата записи.

## 3. Проблемы данных
Проверяем: решения без `####`, дубликаты вопросов, пересечения train и test.

In [ ]:
def norm(s):
    return re.sub(r"[^a-z0-9]+", " ", s.lower()).strip()

tr, te = df[df.split == "train"], df[df.split == "test"]
tr_set, te_set = set(tr.question.map(norm)), set(te.question.map(norm))

print("решений без '####':", int((~df.has_marker).sum()))
print("дубликатов вопросов внутри train:", int(tr.question.map(norm).duplicated().sum()))
print("дубликатов вопросов внутри test:", int(te.question.map(norm).duplicated().sum()))
print("точных пересечений train и test:", len(tr_set & te_set))
print("доля решений с аннотациями калькулятора <<...>>:", round(df.calc_annot.mean(), 3))

Аннотации калькулятора `<<48/2=24>>` попадают в обучающий текст как есть. Это шум для модели и лишние токены. Если хотите их убрать, поставьте `data.strip_calc_annotations: true`, но тогда формат обучения отличается от примеров 5-shot в lm-eval: это нужно упомянуть в работе.

## 4. Разбиение train/val и утечки

`build_datasets` отбрасывает обучающие задачи, у которых есть общие 10-граммы с GSM8K test, потом перемешивает по `data_seed`, отделяет `val_size` примеров на валидацию, а из остатка берёт подвыборку `train_size`.
Проверим три свойства: подвыборки **вложены**, train и val **не пересекаются**, число удалённых по утечке задач.

In [ ]:
train_all, val, report = build_datasets(cfg)
print(json.dumps(report, indent=1, ensure_ascii=False))

subsets = {}
for n in (1000, 3000):
    c = load_config(["configs/base.yaml"], [f"data.train_size={n}"])
    subsets[n], _, _ = build_datasets(c)

q = lambda rows: [r["question"] for r in rows]
assert q(subsets[1000]) == q(train_all)[:1000], "подвыборка 1000 не вложена"
assert q(subsets[3000]) == q(train_all)[:3000], "подвыборка 3000 не вложена"
assert not (set(q(train_all)) & set(q(val))), "train и val пересекаются"
print("OK: подвыборки вложены, train и val не пересекаются")

In [ ]:
# сравнение train и val по основным свойствам: val должен быть из того же распределения
by_q = df[df.split == "train"].set_index("question")
val_df = by_q.loc[by_q.index.intersection([r["question"] for r in val])]
cmp = pd.DataFrame({
    "train (весь)": by_q[["total_tok", "steps"]].mean(),
    "val": val_df[["total_tok", "steps"]].mean(),
}).round(2)
cmp

## 5. (Опционально) MetaMathQA

Нужен только для стадии `meta` (25k примеров). Загрузка большая (~395k строк), поэтому по умолчанию выключена.
Что проверить: лицензию, как получены данные (расширение GSM8K и MATH), формат ответа после преобразования в `#### N`, долю примеров длиннее `max_seq_len`.

In [ ]:
LOAD_META = False

if LOAD_META:
    cfg_m = load_config(["configs/base.yaml"], ["data.train_dataset=metamathqa", "data.train_size=25000"])
    tr_m, val_m, rep_m = build_datasets(cfg_m)
    print(json.dumps(rep_m, indent=1, ensure_ascii=False))
    lens_m = np.array(tok_len([TEMPLATE.format(question=r["question"]) + " " + r["solution"] for r in tr_m]))
    print("доля длиннее max_seq_len:", round(float((lens_m > MAX_LEN).mean()), 4))
    print("без '####':", sum("####" not in r["solution"] for r in tr_m))
    print("\n--- пример ---\nQ:", tr_m[0]["question"], "\nA:", tr_m[0]["solution"])

## 6. Сохранение сводки

In [ ]:
eda = {"gsm8k": summary, "split_report": report}
(ROOT / "results" / "eda_summary.json").write_text(json.dumps(eda, indent=1, ensure_ascii=False, default=str))
print("сохранено в results/eda_summary.json")

## 7. Выводы (заполнить по результатам)

- **Объём:** сколько задач в train/val/test после фильтра; сколько удалено как утечка.
- **Формат:** какая доля решений содержит аннотации калькулятора; что сделано (оставлены/удалены).
- **Длины:** какая доля примеров отброшена по `max_seq_len`; влияет ли это на выбор `max_seq_len`.
- **Распределение:** похожи ли train, val и test по числу шагов, длине и типу ответа.
- **Ограничения:** только текстовые задачи на английском, школьный уровень; ответы с дробями и единицами могут извлекаться неоднозначно.
- **Что передать дальше:** решения, которые попадут в README и `docs/data_card.md`.